# 🚲 Greenness: are greener streets calmer for cyclists?
### Enriching GPS trajectories with Earth Observation, OpenStreetMap and Street View Imagery
*Summer School: Earth Observation and GeoAI for Climate-Resilient Urban Systems*

---

## Your research question
> **Are stress hotspots less green — from above (NDVI) and at eye level (Green View Index)? Can we route cyclists along greener streets?**

You will measure greenness **from space** with Sentinel-2 and **at eye level** with street view photos and AI, compare both views, and finally compute a **green route** with Dijkstra's algorithm.


**How to use:** run cells top to bottom with **Shift + Enter**. In the **✏️ Your turn** tasks, fill in the blanks `____` and run the **✅ check** cell below. Boxes marked **📖 Deep dive** or **🔍 What the code does** are optional.

### 🐍 Python basics
| you see | it means |
|---|---|
| `____` | a blank for you to fill in (four underscores) |
| `df.head()` | first 5 rows of the table `df` |
| `df.column` or `df["column"]` | one column of a table |
| `df[df.speed_kmh < 45]` | keep only the rows where the condition is true |
| `df.groupby("a").b.mean()` | mean of column `b` for every value of column `a` |
| `(cond1) & (cond2)` | both conditions must be true (put each in brackets!) |
| `f"{x:.2f}"` | text with the value of `x` rounded to 2 decimals |


## 0 · Setup 
Run the next three cells. They install a few packages, set the options and load the data and helper functions.

<details><summary>🔍 Which packages are installed?</summary>

| package | purpose |
|---|---|
| `osmnx` | download OpenStreetMap data |
| `pystac-client`, `planetary-computer`, `odc-stac` | search and load satellite images from Microsoft's Planetary Computer (STAC catalogue) |
| `mapclassify` | quantile classes for map colours |
| `h3` | Uber's hexagonal grid (optional alternative to square cells) |
| `ultralytics` | YOLO26 object detection |
</details>

In [ ]:
# Install the few packages Colab does not ship with (takes ~1 min)
!pip install -q osmnx pystac-client planetary-computer odc-stac mapclassify h3 ultralytics

<details><summary>⚙️ Configuration options (defaults work — change only if you want to)</summary>

- `LIVE`: `True` = try fresh downloads first; `False` = use only the pre-downloaded data. Any failed download automatically falls back to the pre-downloaded data.
- `MAPILLARY_TOKEN` *(optional)*: needed only to download street view images live ([get a free client token](https://www.mapillary.com/dashboard/developers), starts with `MLY|`).
- `CARTO_API_KEY` *(optional)*: free key for the light-grey CARTO background map; without it, maps use OpenStreetMap tiles.
- `GRID_TYPE`: `"square"` (100 m cells) or `"h3"` (hexagons, `H3_RES = 10` ≈ 13 000 m²). Try both after the workshop to see the MAUP in action.
- `DATA_URL`: direct download link of the workshop data.
</details>

In [ ]:
# ⚙️ Configuration
LIVE = True            # try live downloads first; set False to use only the pre-downloaded fallback
MAPILLARY_TOKEN = ""  # optional: your client token from https://www.mapillary.com/dashboard/developers (starts with "MLY|")
CARTO_API_KEY = ""       # optional: free key from https://carto.com/basemaps/apikey/ for the light-grey CARTO background map

GRID_TYPE = "square"   # spatial unit: "square" (100 m x 100 m cells) or "h3" (hexagons)
CELL = 100             # side length of square cells in metres          (used if GRID_TYPE == "square")
H3_RES = 10            # H3 resolution: 9 ≈ 105 000 m², 10 ≈ 13 000 m², 11 ≈ 1 900 m²  (used if GRID_TYPE == "h3")

# Where the workshop data (GPS subset + fallbacks) is hosted: a DIRECT download link to the zip.
# Alternatively upload workshop_data.zip via the 📁 file panel on the left.
DATA_URL = "https://raw.githubusercontent.com/mms1068467/IEEE_GRSS_multimodal_mobility_data/main/workshop_data.zip"

The next cell downloads the data, defines the 4 × 4 km study area around Osnabrück's centre and a few helper functions. You do not need to read it.

<details><summary>🔍 What the code does</summary>

- **CRS:** GPS gives degrees (`EPSG:4326`), which are not a distance unit. All data is projected to **ETRS89 / UTM 32N (`EPSG:25832`)**, in metres.
- **`get_data(name, fetch, kind)`** tries the live download `fetch()`; if it fails, it loads `workshop_data/<name>` instead.
- **`cell_id(x, y)`** gives the grid cell of a point (square: `"column_row"`, e.g. `"4348_57922"`; H3: a hexagon ID).
- **`cells_to_gdf(df)`** turns a table indexed by cell ID into polygons for mapping.
- **`raster_to_cells(da, name)`** averages all raster pixels inside each cell (**zonal mean**).
- **`show_map(gdf, column)`** draws an interactive map with a switchable satellite background.
</details>

In [ ]:
import os, zipfile, urllib.request, warnings, json
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import folium, xyzservices.providers as xyz
from shapely.geometry import box
warnings.filterwarnings("ignore")

# ======================= 1. get the workshop data =======================
DATA_DIR = Path("workshop_data")

def _direct_link(url):
    """Turn common 'share page' links into direct file downloads."""
    if "github.com" in url and "/blob/" in url: url = url.replace("/blob/", "/raw/")
    if "dropbox.com" in url: url = url.replace("dl=0", "dl=1")
    return url

def _extract(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        # works whether the zip contains the folder 'workshop_data/' or just the files
        z.extractall("." if any(n.startswith("workshop_data/") for n in names) else DATA_DIR)

def get_workshop_data():
    if (DATA_DIR / "gps_osnabrueck.parquet").exists():
        return
    DATA_DIR.mkdir(exist_ok=True)
    if Path("gps_osnabrueck.parquet").exists():                 # parquet uploaded on its own
        Path("gps_osnabrueck.parquet").rename(DATA_DIR / "gps_osnabrueck.parquet"); return
    for z in sorted(Path(".").glob("workshop_data*.zip")):      # zip uploaded via Colab's file panel
        if zipfile.is_zipfile(z): _extract(z); return
        z.unlink()                                               # remove broken/partial downloads
    if "<your-account>" in DATA_URL:
        raise RuntimeError("Set DATA_URL in the configuration cell, or upload workshop_data.zip "
                           "via the 📁 file panel on the left, then re-run this cell.")
    url = _direct_link(DATA_URL)
    print("Downloading workshop data …")
    if "drive.google.com" in url:                               # Google Drive share links need gdown
        import gdown; gdown.download(url, "workshop_data.zip", quiet=False, fuzzy=True)
    else:
        urllib.request.urlretrieve(url, "workshop_data.zip")
    if not zipfile.is_zipfile("workshop_data.zip"):
        head = Path("workshop_data.zip").read_bytes()[:150]
        Path("workshop_data.zip").unlink()
        raise RuntimeError("DATA_URL did not return a zip file (it probably points to a web page, "
                           f"not the file itself). First bytes received: {head!r}")
    _extract("workshop_data.zip")

get_workshop_data()
print("Data folder:", sorted(p.name for p in DATA_DIR.iterdir()))

# ======================= 2. study area: 4 x 4 km around Osnabrück's city centre =======================
CRS = "EPSG:25832"                      # ETRS89 / UTM zone 32N – metres, official CRS for western Germany
X0, Y0, HALF = 434750, 5792250, 2000    # centre (easting, northing) and half side length in metres
AREA = gpd.GeoDataFrame(geometry=[box(X0-HALF, Y0-HALF, X0+HALF, Y0+HALF)], crs=CRS)
BBOX_WGS84 = tuple(AREA.to_crs(4326).total_bounds)      # (west, south, east, north) in degrees

# ======================= 3. helper functions =======================
# ---------- live download with automatic fallback ----------
def _save(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(obj, gpd.GeoDataFrame): obj.to_parquet(path)
    elif isinstance(obj, pd.DataFrame):   obj.to_parquet(path)
    else:  # xarray.Dataset -> table of pixels
        obj.to_dataframe().reset_index()[["y", "x", *obj.data_vars]].to_parquet(path)

def _load(path, kind):
    if kind == "gdf":    return gpd.read_parquet(path)
    if kind == "df":     return pd.read_parquet(path)
    if kind == "raster": return pd.read_parquet(path).set_index(["y", "x"]).to_xarray()

def get_data(name, fetch, kind):
    """Try the live download; on any problem fall back to workshop_data/<name>."""
    path = DATA_DIR / name
    if LIVE:
        try:
            obj = fetch()
            _save(obj, path)
            print(f"✅ {name}: live download OK")
            return obj
        except Exception as e:
            print(f"⚠️ {name}: live download failed ({type(e).__name__}: {str(e)[:120]}) → fallback")
    obj = _load(path, kind)
    print(f"📦 {name}: loaded fallback")
    return obj

# ---------- grid helpers (square cells or H3 hexagons, see GRID_TYPE) ----------
assert GRID_TYPE in ("square", "h3"), 'GRID_TYPE must be "square" or "h3"'
if GRID_TYPE == "h3":
    import h3
    from pyproj import Transformer
    from shapely.geometry import Polygon
    _to_wgs84 = Transformer.from_crs(CRS, 4326, always_xy=True)

def cell_id(x, y):
    """ID of the grid cell that contains (x, y) given in EPSG:25832."""
    if GRID_TYPE == "square":   # 'column_row', e.g. '4348_57922'
        return (np.floor(np.asarray(x) / CELL).astype(int).astype(str).astype(object) + "_" +
                np.floor(np.asarray(y) / CELL).astype(int).astype(str).astype(object))
    lon, lat = _to_wgs84.transform(np.asarray(x), np.asarray(y))      # H3 works on lat/lon
    return np.array([h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(lat, lon)], dtype=object)

def cells_to_gdf(df):
    """Turn a table indexed by cell_id into polygons (squares or hexagons)."""
    if GRID_TYPE == "square":
        ij = df.index.to_series().str.split("_", expand=True).astype(int)
        geom = [box(i*CELL, j*CELL, (i+1)*CELL, (j+1)*CELL) for i, j in zip(ij[0], ij[1])]
        return gpd.GeoDataFrame(df, geometry=geom, crs=CRS)
    geom = [Polygon([(lo, la) for la, lo in h3.cell_to_boundary(c)]) for c in df.index]
    return gpd.GeoDataFrame(df, geometry=geom, crs=4326).to_crs(CRS)

CELL_LABEL = f"{CELL} m square" if GRID_TYPE == "square" else f"H3 res {H3_RES} hexagon"
print("Spatial unit:", CELL_LABEL)

def raster_to_cells(da, name):
    """Average all pixels of a raster (xarray.DataArray in EPSG:25832) per grid cell (zonal mean)."""
    t = da.to_dataframe(name).reset_index()[["x", "y", name]].dropna()
    t["cell_id"] = cell_id(t.x, t.y)              # pixel centre -> cell it falls into
    return t.groupby("cell_id")[name].mean()

# ---------- interactive maps ----------
# CARTO tiles need a (free) API key; without one we fall back to the standard OpenStreetMap tiles
BASEMAP = xyz.CartoDB.Positron(apikey=CARTO_API_KEY) if CARTO_API_KEY else xyz.OpenStreetMap.Mapnik

def show_map(gdf, column, **kwargs):
    """Interactive choropleth map with a switchable satellite background."""
    m = gdf.explore(column=column, tiles=BASEMAP, **kwargs)
    folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m)
    folium.LayerControl().add_to(m)
    return m

print("Study area bbox (WGS84):", np.round(BBOX_WGS84, 4))

In [ ]:
def check(condition, ok="✅ Well done!", hint="❌ Not quite yet – look at the hints above the task."):
    """Friendly feedback for the ✏️ Your turn tasks."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        passed, hint = False, f"❌ Not finished yet ({type(e).__name__}) – fill in all blanks `____` and run the task cell first."
    print(ok if passed else hint)

def xr_corr(a, b):
    """Correlation of two rasters (ignoring missing pixels)."""
    m = np.isfinite(a.values) & np.isfinite(b.values)
    return np.corrcoef(a.values[m], b.values[m])[0, 1]


## 1 · Load & explore the stress trajectories 

- A **trajectory** is a time-ordered sequence of points $(x_i, y_i, t_i)$ — here **one point per second (1 Hz)**, with speed and heading. GPS error is about **3–5 m** under open sky and up to tens of metres between tall buildings.


| column | meaning |
|---|---|
| `rider_id`, `ride_id` | pseudonymous IDs of person and trip |
| `time_utc`, `lat`, `lon` | timestamp (UTC) and position (WGS84) |
| `speed_kmh`, `bearing` | GPS speed and heading |
| `mos_score` | stress score from EDA (0–2) |
| `detected_mos` | 1 = Moment of Stress |

The data is pseudonymised, and the first/last 200 m of each ride were removed to protect home locations.

In [ ]:
gps = pd.read_parquet(DATA_DIR / "gps_osnabrueck.parquet")                    # 1. read the table
gps = gpd.GeoDataFrame(gps, geometry=gpd.points_from_xy(gps.lon, gps.lat),    # 2. make points (degrees)
                       crs=4326).to_crs(CRS)                                  # 3. reproject to metres
gps["x"], gps["y"] = gps.geometry.x, gps.geometry.y                           # metric coordinates as columns

print(f"{len(gps):,} GPS points · {gps.ride_id.nunique()} rides · {gps.rider_id.nunique()} riders")
print("period:", gps.time_utc.min().date(), "→", gps.time_utc.max().date())
gps.head()

**Quick look at the data.** 

What do you notice about speed (peak at 0?), the stress score and the months of data collection?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
gps.speed_kmh.clip(upper=60).hist(bins=60, ax=ax[0]); ax[0].set_title("speed (km/h)")
gps.mos_score.value_counts().sort_index().plot.bar(ax=ax[1]); ax[1].set_title("mos_score (log scale)"); ax[1].set_yscale("log")
gps.time_utc.dt.to_period("M").value_counts().sort_index().plot.bar(ax=ax[2]); ax[2].set_title("points per month")
plt.tight_layout()

**Cleaning.** We drop implausible speeds (> 45 km/h: GPS errors or a ride in a car or bus) and flag points below 2 km/h as `stopped` (waiting at lights, pushing the bike) — we keep them, because waiting may itself be stressful.

In [ ]:
n0 = len(gps)
gps = gps[gps.speed_kmh < 45].copy()     # drop implausible speeds
gps["stopped"] = gps.speed_kmh < 2      # True/False column: is the cyclist (almost) standing still?
print(f"removed {n0 - len(gps):,} implausible points; {gps.stopped.mean():.0%} of remaining points are stops")

### ✏️ Your turn 1
Is stress more frequent when cyclists are **stopped** or **moving**?

Every row is one second of riding. `detected_mos` is 1 for a stressed second and 0 otherwise, so the **mean** of `detected_mos` is the *share* of stressed seconds. Multiply that share by 3600 (seconds per hour) to get **stress events per hour**.

Fill in the blanks: group by the column that says whether a cyclist is stopped, take the mean, multiply by 3600.

<details><summary>🆘 Hint</summary>

The column created in the cleaning step is called `stopped`. The function for the average is `.mean()`.
</details>

In [ ]:
stress_per_hour = gps.groupby("____").detected_mos.____() * ____
stress_per_hour

In [ ]:
check(lambda: set(stress_per_hour.index) == {True, False} and 0 < stress_per_hour.max() < 3600,
      ok="✅ Well done! Which group has more stress events per hour?")

> 💬 **Point for discussion:** Why could stopping go together with stress? Think of traffic lights, conflicts with cars at junctions, etc.

## 2 · From points to places: a grid of cells 

Our points describe **people**; we want to learn about **places**. We therefore aggregate all points inside the same **100 m × 100 m cell**. This smooths out GPS error and the delay in physiological signals, turns rare events into stable rates, and gives us a common unit for attaching satellite, OSM and street view data.

Our outcome is the **stress rate** — Moments of Stress per hour of riding in a cell:

We divide by the time spent in the cell (the **exposure**) because a simple count would mostly measure how many people ride there.

<details><summary>🔍 What the code does</summary>

`cell_id` assigns each point to a cell; `groupby("cell_id").agg(...)` computes one row per cell: seconds of riding (`n_points`), distinct rides and riders, mean stress score, stress rate, median speed and share of time stopped.
</details>

In [ ]:
gps["cell_id"] = cell_id(gps.x, gps.y)            # assign each point to its grid cell (square or H3)

cells = gps.groupby("cell_id").agg(
    n_points    = ("ride_id", "size"),             # seconds of riding in the cell
    n_rides     = ("ride_id", "nunique"),          # distinct trips
    n_riders    = ("rider_id", "nunique"),         # distinct people
    mos_mean    = ("mos_score", "mean"),           # mean stress score
    stress_rate = ("detected_mos", lambda s: s.mean() * 3600),   # stress events per hour of riding (see above)
    speed_med   = ("speed_kmh", "median"),
    stop_share  = ("stopped", "mean"),             # share of time standing still
)
print(len(cells), f"cells ({CELL_LABEL}) with data")
cells.describe().round(2)

### ✏️ Your turn 2
A cell visited by one rider on one day tells us more about that **person** than about the **place**.
Keep only cells with **at least 10 rides and at least 3 different riders**.

<details><summary>🆘 Hint</summary>

"At least" means `>=`. The columns are `n_rides` and `n_riders`; the thresholds are stored in `MIN_RIDES` and `MIN_RIDERS`. Both conditions must be true: `(condition 1) & (condition 2)`.
</details>

In [ ]:
MIN_RIDES, MIN_RIDERS = 10, 3
cells_original = cells.copy()                      # keep a copy for comparison
cells = cells[(cells.n_rides >= ____) & (cells.____ >= MIN_RIDERS)]
print(len(cells), "reliable cells out of", len(cells_original), f"({len(cells)/len(cells_original):.0%} kept)")

In [ ]:
check(lambda: len(cells) < len(cells_original) and (cells.n_rides >= 10).all() and (cells.n_riders >= 3).all())

**Stress map.** Colours use **quantiles** (each of the 5 classes holds 20 % of the cells) — check the legend for the real values. Switch to the satellite background with the layer button (top right).

In [ ]:
cells_gdf = cells_to_gdf(cells)
show_map(cells_gdf, "stress_rate", cmap="Reds", scheme="quantiles", k=5,
         legend_kwds={"caption": "stress events / hour"})

> 💬 **Point for discussion:** Where do stress hotspots occur — junctions, main roads, the old town? How would the picture change with 50 m or 250 m cells (MAUP)?

## 3 · Sentinel-2: greenness and built-up from above 

 We use the **Sentinel-2** Level-2A **surface reflectance** product, mask clouds with its scene classification layer and take the per-pixel **median** of the clearest summer-2023 scenes.

Vegetation absorbs **red** light but strongly reflects **near-infrared (NIR)**; built-up surfaces reflect more **short-wave infrared (SWIR)** than NIR:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}} \qquad\qquad \text{NDBI} = \frac{\text{SWIR} - \text{NIR}}{\text{SWIR} + \text{NIR}}$$

NDVI: < 0 water · 0–0.2 sealed surfaces · 0.2–0.5 sparse vegetation · > 0.6 dense vegetation.

<details><summary>📖 Deep dive: bands</summary>

| band | name | wavelength | resolution |
|---|---|---|---|
| B04 | red | 665 nm | 10 m |
| B08 | NIR | 842 nm | 10 m |
| B11 | SWIR | 1610 nm | 20 m |

</details>

In [ ]:
def fetch_sentinel2():
    import pystac_client, planetary_computer, odc.stac, xarray as xr
    # 1. open the STAC catalogue
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                        modifier=planetary_computer.sign_inplace)
    # 2. search scenes: our area, summer 2023, < 20 % clouds
    items = catalog.search(collections=["sentinel-2-l2a"], bbox=BBOX_WGS84,
                           datetime="2023-06-01/2023-08-31",
                           query={"eo:cloud_cover": {"lt": 20}}).item_collection()
    items = sorted(items, key=lambda i: i.properties["eo:cloud_cover"])[:8]     # 8 least cloudy scenes
    print(f"using {len(items)} scenes:", [i.datetime.date().isoformat() for i in items])
    # 3. load only our study area, reprojected to EPSG:25832 at 10 m
    ds = odc.stac.load(items, bands=["B04", "B08", "B11", "SCL"], bbox=BBOX_WGS84,
                       crs=CRS, resolution=10, groupby="solar_day")
    # 4. clear-sky mask: vegetation, not vegetated, water, unclassified
    clear = ds.SCL.isin([4, 5, 6, 7])
    # 5. digital numbers -> reflectance, mask clouds, median over time
    out = {}
    for band, name in [("B04", "red"), ("B08", "nir"), ("B11", "swir")]:
        refl = (ds[band].astype("float32") - 1000) / 10000   # L2A baseline ≥ 04.00 has a -1000 offset
        out[name] = refl.where(clear & (ds[band] > 0)).median("time").clip(0, 1)
    return xr.Dataset(out)

s2 = get_data("s2_summer2023.parquet", fetch_sentinel2, "raster")
s2

`s2` is an **`xarray.Dataset`** of labelled 2-D arrays (`red`, `nir`, `swir`). Arithmetic works pixel by pixel, so NDVI is written exactly like the formula:

In [ ]:
s2["ndvi"] = (s2.nir - s2.red) / (s2.nir + s2.red)        # pixel-wise, like the formula above
s2.ndvi.plot(cmap="RdYlGn", vmin=-0.2, vmax=0.9, figsize=(6, 5)); plt.title("NDVI, summer 2023 median");

### ✏️ Your turn 3
Compute the **built-up index** NDBI = (SWIR − NIR) / (SWIR + NIR) and plot it — just like NDVI in the cell above, but with other bands.
The bands are stored as `s2.swir`, `s2.nir` and `s2.red`. Where does the NDBI map disagree with the NDVI map?

<details><summary>🆘 Hint</summary>

Look at the NDVI cell above: `(s2.nir - s2.red) / (s2.nir + s2.red)`. For NDBI the first band is SWIR and the second is NIR.
</details>

In [ ]:
s2["ndbi"] = (s2.____ - s2.____) / (s2.____ + s2.____)
s2.ndbi.plot(cmap="RdGy_r", vmin=-0.5, vmax=0.3, figsize=(6, 5)); plt.title("NDBI, summer 2023 median");

In [ ]:
check(lambda: "ndbi" in s2 and float(s2.ndbi.mean()) < 0.3 and float(xr_corr(s2.ndbi, s2.ndvi)) < 0,
      ok="✅ Well done! Built-up and green areas are (mostly) opposites.")

**From pixels to cells (zonal statistics).** Each 100 m cell contains 10 × 10 Sentinel-2 pixels; `raster_to_cells` averages them.

In [ ]:
cells["ndvi"] = raster_to_cells(s2.ndvi, "ndvi")
if "ndbi" in s2:
    cells["ndbi"] = raster_to_cells(s2.ndbi, "ndbi")
else:
    print("⚠️ NDBI not found – finish Your turn 3 and re-run this cell. Continuing without NDBI.")
cells[[c for c in ["stress_rate", "ndvi", "ndbi"] if c in cells]].describe().round(2)

**Greenness map.** Where are the greenest and the most built-up cells? Use the layer button to compare with the satellite image.

In [ ]:
show_map(cells_to_gdf(cells), "ndvi", cmap="RdYlGn", tooltip=[c for c in ["ndvi", "ndbi", "stress_rate"] if c in cells])

## 4 · Street view: greenness at eye level 

Satellites see roofs and tree tops; cyclists look **ahead** and see façades, cars, trees and sky. **Mapillary** is a crowd-sourced street view platform (images © Mapillary contributors, CC-BY-SA). We take one photo for each of the **15 "calmest"** and **15 most stressful** cells.

Two pre-trained deep-learning models analyse the photos:
- **Semantic segmentation** (SegFormer, trained on the Cityscapes street scenes) gives **a class for every pixel** → how much of the view is vegetation, sky, cars …
- **Object detection** (YOLO26, trained on the 80 COCO classes) draws **boxes** around individual objects such as traffic lights.

$$\text{Green View Index (GVI)} = \frac{\text{vegetation + terrain pixels}}{\text{all pixels}}$$

GVI (Li et al. 2015) measures *perceived* greenness at eye level.

<details><summary>📖 Deep dive: models and biases</summary>

| task | output | example question |
|---|---|---|
| image classification | one label per image | "Is this a residential street?" |
| object detection | boxes around objects | "How many cars are there?" |
| semantic segmentation | a class for every pixel | "How much of the view is vegetation?" |

- **SegFormer** (Xie et al. 2021) is a transformer-based segmentation network; we use the small `b0` variant (3.7 M parameters, runs on a CPU). Cityscapes (Cordts et al. 2016) contains street scenes from 50 mostly German cities — very similar to Osnabrück.
- **YOLO** ("You Only Look Once") predicts all boxes in a single pass; we use `yolo26s` at 1024 px because traffic lights are small. Each box has a **confidence**; we keep boxes with `CONF ≥ 0.35`.
- **Biases:** images exist where contributors went; different years and seasons (leafless trees!); a car camera sees the street differently than a cyclist; one image per cell is a small sample; models perform worse at night or in rain.
- References: Li et al. (2015); Xie et al. (2021); Cordts et al. (2016); Biljecki & Ito (2021).
</details>

<details><summary>🔍 What the code does</summary>

1. Take the 15 lowest and 15 highest stress cells (a contrast design that needs few images).
2. For each cell, ask the Mapillary API for images inside the cell, skip 360° panoramas and keep the most recent photo.
3. Segment every photo with SegFormer, compute the share of each class, and run YOLO26.
Without a Mapillary token, pre-downloaded images and results are used. With live processing, a GPU (*Runtime → Change runtime type → T4 GPU*) speeds things up.
</details>

In [ ]:
ranked = cells.sort_values("stress_rate")
svi_cells = pd.concat([ranked.head(15).assign(group="low stress"),       # 15 calmest cells
                       ranked.tail(15).assign(group="high stress")])     # 15 most stressful cells
svi_gdf = cells_to_gdf(svi_cells)
SVI_DIR = DATA_DIR / "svi"; SVI_DIR.mkdir(exist_ok=True)

def fetch_mapillary():
    import requests
    if not MAPILLARY_TOKEN: raise RuntimeError("no Mapillary token set")
    rows = []
    for cid, geom in svi_gdf.to_crs(4326).geometry.items():
        # all images inside the cell (bbox = west,south,east,north in degrees)
        r = requests.get("https://graph.mapillary.com/images", timeout=30, params={
            "access_token": MAPILLARY_TOKEN, "bbox": ",".join(map(str, geom.bounds)), "limit": 100,
            "fields": "id,captured_at,compass_angle,is_pano,thumb_1024_url,computed_geometry,geometry"})
        r.raise_for_status()
        imgs = [i for i in r.json().get("data", []) if not i.get("is_pano")]   # skip 360° panoramas
        if not imgs: continue
        img = max(imgs, key=lambda i: i["captured_at"])            # most recent photo
        (SVI_DIR / f"{cid}.jpg").write_bytes(requests.get(img["thumb_1024_url"], timeout=30).content)
        lon, lat = (img.get("computed_geometry") or img["geometry"])["coordinates"]
        rows.append({"cell_id": cid, "image_id": img["id"], "compass_angle": img.get("compass_angle"),
                     "captured_at": pd.to_datetime(img["captured_at"], unit="ms"), "lon": lon, "lat": lat})
    meta = pd.DataFrame(rows)
    return gpd.GeoDataFrame(meta, geometry=gpd.points_from_xy(meta.lon, meta.lat), crs=4326)

svi = get_data(f"svi_images_{GRID_TYPE}.parquet", fetch_mapillary, "gdf")   # one file per grid type
svi = svi[svi.cell_id.isin(svi_cells.index) & svi.cell_id.map(lambda c: (SVI_DIR / f"{c}.jpg").exists())]
print(len(svi), "cells with a street view image")

In [ ]:
from PIL import Image
segmenter = None

# the 19 Cityscapes classes and their official colours (used for plotting)
CITYSCAPES = ["road", "sidewalk", "building", "wall", "fence", "pole", "traffic light", "traffic sign",
              "vegetation", "terrain", "sky", "person", "rider", "car", "truck", "bus", "train",
              "motorcycle", "bicycle"]
PALETTE = np.array([(128, 64, 128), (244, 35, 232), (70, 70, 70), (102, 102, 156), (190, 153, 153),
                    (153, 153, 153), (250, 170, 30), (220, 220, 0), (107, 142, 35), (152, 251, 152),
                    (70, 130, 180), (220, 20, 60), (255, 0, 0), (0, 0, 142), (0, 0, 70), (0, 60, 100),
                    (0, 80, 100), (0, 0, 230), (119, 11, 32)], dtype=np.uint8)

def segment_image(path):
    """Label map: every pixel gets the index of its Cityscapes class (255 = unlabelled)."""
    global segmenter
    if segmenter is None:   # load the model once (~15 MB from Hugging Face)
        from transformers import pipeline
        segmenter = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-cityscapes-1024-1024")
    img = Image.open(path).convert("RGB")
    label = np.full((img.height, img.width), 255, dtype=np.uint8)
    for m in segmenter(img):                                   # one binary mask per class found
        if m["label"] in CITYSCAPES:
            label[np.array(m["mask"]) > 0] = CITYSCAPES.index(m["label"])
    return label

def segment_all():
    rows = []
    for c in svi.cell_id:
        label = segment_image(SVI_DIR / f"{c}.jpg")
        Image.fromarray(label).save(SVI_DIR / f"{c}_seg.png")                  # keep for visualisation
        rows.append({"cell_id": c, **{name: (label == k).mean() for k, name in enumerate(CITYSCAPES)}})
    return pd.DataFrame(rows)

seg = get_data(f"svi_segmentation_{GRID_TYPE}.parquet", segment_all, "df").set_index("cell_id")
for c in ["vegetation", "terrain", "sky", "road", "car", "building"]:   # classes absent from every image → 0
    if c not in seg: seg[c] = 0.0
seg["gvi"] = seg.vegetation + seg.terrain                                 # Green View Index
seg = seg.join(svi_cells[["group", "stress_rate", "ndvi"]], how="inner")  # add stress and NDVI of the cell
seg[["group", "gvi", "sky", "road", "car", "building"]].head()

In [ ]:
CONF = 0.35                     # minimum confidence for a detection to count
detector = None

def detect_all():
    global detector
    from ultralytics import YOLO
    if detector is None: detector = YOLO("yolo26s.pt")         # latest Ultralytics model, COCO classes
    rows = []
    for c in svi.cell_id:
        r = detector.predict(str(SVI_DIR / f"{c}.jpg"), conf=0.1, imgsz=1024, verbose=False)[0]
        for (x1, y1, x2, y2), k, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            rows.append({"cell_id": c, "label": r.names[int(k)], "conf": p, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return pd.DataFrame(rows, columns=["cell_id", "label", "conf", "x1", "y1", "x2", "y2"])

det = get_data(f"svi_detections_{GRID_TYPE}.parquet", detect_all, "df")
det = det[det.cell_id.isin(seg.index) & (det.conf >= CONF)]        # apply the confidence threshold
print(det.label.value_counts().head(8))

**Original · segmentation · detection** for the calmest and the most stressful cell (traffic lights in orange):

In [ ]:
from matplotlib.patches import Patch, Rectangle

def show_triptych(cid):
    img = Image.open(SVI_DIR / f"{cid}.jpg").convert("RGB")
    seg_png = SVI_DIR / f"{cid}_seg.png"
    label = np.array(Image.open(seg_png)) if seg_png.exists() else segment_image(SVI_DIR / f"{cid}.jpg")
    colour = np.zeros((*label.shape, 3), dtype=np.uint8)
    colour[label < 255] = PALETTE[label[label < 255]]

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    ax[0].imshow(img); ax[0].set_title("original street view image")
    ax[1].imshow(img); ax[1].imshow(colour, alpha=0.75); ax[1].set_title("semantic segmentation (SegFormer)")
    shares = pd.Series({n: (label == k).mean() for k, n in enumerate(CITYSCAPES)})
    ax[1].legend(handles=[Patch(color=PALETTE[CITYSCAPES.index(n)] / 255, label=f"{n} {v:.0%}")
                          for n, v in shares[shares > 0.01].sort_values(ascending=False).items()],
                 loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8, frameon=False)
    ax[2].imshow(img); ax[2].set_title(f"object detection (YOLO26, conf ≥ {CONF})")
    for _, b in det[det.cell_id == cid].iterrows():
        col = "orange" if b.label == "traffic light" else "cyan"
        ax[2].add_patch(Rectangle((b.x1, b.y1), b.x2 - b.x1, b.y2 - b.y1, fill=False, color=col, lw=2))
        ax[2].text(b.x1, b.y1 - 3, f"{b.label} {b.conf:.2f}", color="black", fontsize=7,
                   bbox=dict(facecolor=col, alpha=.8, pad=1, lw=0))
    for a in ax: a.axis("off")
    fig.suptitle(f"cell {cid} · {seg.loc[cid, 'group']} · {seg.loc[cid, 'stress_rate']:.0f} stress events/h · "
                 f"GVI {seg.loc[cid, 'gvi']:.0%}   (image © Mapillary contributors, CC-BY-SA)", y=1.02)
    plt.tight_layout(); plt.show()

show_triptych(seg.stress_rate.idxmin())    # calmest cell with an image
show_triptych(seg.stress_rate.idxmax())    # most stressful cell with an image

### ✏️ Your turn 4
Compare the **mean** green view (`gvi`), sky, cars and NDVI of the *low stress* and *high stress* cells.
Then look at the plot: do eye-level greenness (GVI) and top-down greenness (NDVI) tell the same story?

<details><summary>🆘 Hint</summary>

The column that says *low stress* / *high stress* is called `group`. The function for the average is `.mean()`.
</details>

In [ ]:
group_means = seg.groupby("____")[["gvi", "sky", "car", "ndvi"]].____().round(3)
print(group_means)
ax = seg.plot.scatter("ndvi", "gvi", c=seg.group.map({"low stress": "tab:blue", "high stress": "tab:red"}), figsize=(5, 4))
ax.set_title(f"NDVI vs GVI · Spearman r = {seg[['ndvi', 'gvi']].corr('spearman').iloc[0, 1]:.2f}");

In [ ]:
check(lambda: set(group_means.index) == {"low stress", "high stress"} and "gvi" in group_means)

### ✏️ Your turn 5
Do satellite and street view agree? Compute a **greenness mismatch** for every street view cell: rescale both indices to 0–1 and subtract **top-down (NDVI)** from **eye-level (GVI)**.
Positive = greener at eye level than seen from space; negative = green from space (e.g. back gardens) but grey on the street.

<details><summary>🆘 Hint</summary>

The eye-level column is `gvi`, the top-down column is `ndvi`. Eye-level comes first: `scale01(seg.gvi) - ...`.
</details>

In [ ]:
def scale01(s):
    return (s - s.min()) / (s.max() - s.min())       # rescale to 0–1

seg["mismatch"] = scale01(seg.____) - scale01(seg.____)
seg.sort_values("mismatch")[["group", "gvi", "ndvi", "mismatch"]].round(2)

In [ ]:
check(lambda: "mismatch" in seg and seg.mismatch.between(-1, 1).all() and seg[["mismatch", "gvi"]].corr().iloc[0, 1] > 0,
      ok="✅ Well done! Look at the most negative and most positive cells on the map below.",
      hint="❌ Not quite – eye-level (gvi) minus top-down (ndvi).")

In [ ]:
show_map(cells_to_gdf(seg[["group", "gvi", "ndvi", "mismatch", "stress_rate"]]), "mismatch", cmap="RdYlGn",
         style_kwds=dict(weight=1, fillOpacity=0.8), legend_kwds={"caption": "greenness mismatch (GVI − NDVI)"})

> 💬 **Point for discussion:** Which greenness matters more for a cyclist? How can the extraction pipeline be improved?

## 5 · Putting it together 

Which of our features go together with stress? We use **Spearman's ρ** as it correlates the *ranks* of the values, so it captures any monotonic relation ("the more A, the more B") and is robust to skewed data. ρ = +1 perfectly increasing, 0 none, −1 perfectly decreasing; roughly, |ρ| < 0.1 is negligible and |ρ| > 0.3 moderate.

Street-view greenness exists only for 30 cells — compare it with stress using the group means from Part 4.


In [ ]:
features = ['ndvi', 'ndbi', 'stop_share', 'speed_med']
features = [f for f in features if f in cells]   # skips columns of unfinished tasks
corr = cells[features + ["stress_rate", "mos_mean"]].corr(method="spearman")[["stress_rate", "mos_mean"]].loc[features]
corr.sort_values("stress_rate").plot.barh(figsize=(7, 4), color=["tab:red", "tab:orange"])
plt.axvline(0, color="k", lw=.8); plt.title(f"Spearman correlation with stress (per {CELL_LABEL})"); plt.tight_layout()
corr.round(2)

The enriched grid — hover over a cell to see its values. Change `"ndvi"` to `"stress_rate"` to compare the patterns.

In [ ]:
show_map(cells_to_gdf(cells), "ndvi", cmap="RdYlGn",
         tooltip=[c for c in ["stress_rate"] + ['ndvi', 'ndbi', 'stop_share', 'speed_med'] if c in cells])

## 6 · Green routing 

Can our results help cyclists choose a better route? A street network is a **graph**: intersections are **nodes**, street segments are **edges** with a **cost**. **Dijkstra's algorithm** finds the path with the lowest total cost between two points. The algorithm never changes — only the **edge cost** does:

| route | edge cost |
|---|---|
| 🔵 Shortest | `length` |
| 🟢 Greenest | `length × (1 + weight × penalty)` |


<details><summary>🔍 What the code does</summary>

1. Download the cycling street network with `osmnx` (or load the saved copy) and project it to metres.
2. Copy `ndvi` and `stress_rate` from the grid cells to the street segments; `scale01` rescales values to 0–1.
3. Compute the **penalty** of every segment and the cost `rq_cost`; `add_costs(weight)` rebuilds the routable graph.
4. `compare_routes()` runs `nx.shortest_path` for both costs and reports length, detour and the mean NDVI and stress rate along each route.
</details>

In [ ]:
import osmnx as ox, networkx as nx
from shapely.geometry import Point, LineString

GRAPH_FILE = DATA_DIR / "osm_bike_graph.graphml"

def load_street_graph():
    # live download with fallback to the saved copy (same idea as get_data)
    if LIVE or not GRAPH_FILE.exists():
        try:
            G = ox.graph_from_bbox(bbox=BBOX_WGS84, network_type="bike", simplify=True)
            ox.save_graphml(G, GRAPH_FILE)
            print("✅ street network: live download OK"); return G
        except Exception as e:
            print(f"⚠️ street network: live download failed ({type(e).__name__}) → fallback")
    G = ox.load_graphml(GRAPH_FILE); print("📦 street network: loaded fallback"); return G

G = ox.project_graph(load_street_graph(), to_crs=CRS)
nodes, edges = ox.graph_to_gdfs(G)

# every street segment takes the values of the grid cell its centre point lies in
JOIN_COLS = [c for c in ['ndvi', 'stress_rate'] if c in cells]
centres = gpd.GeoDataFrame(geometry=edges.geometry.centroid, crs=CRS)
joined = gpd.sjoin(centres, cells_to_gdf(cells)[JOIN_COLS + ["geometry"]], how="left", predicate="within")
for col in JOIN_COLS:
    edges[col] = joined[col].reindex(edges.index).fillna(cells[col].mean())   # outside reliable cells → mean

def scale01(s):
    """Rescale to 0–1 (5th percentile → 0, 95th percentile → 1)."""
    lo, hi = s.quantile(.05), s.quantile(.95)
    return ((s - lo) / (hi - lo)).clip(0, 1)

edges["penalty"] = 1 - scale01(edges["ndvi"])        # 0 = very green, 1 = grey

def add_costs(weight=1.0):
    # cost = length × (1 + weight × penalty): with weight 0 every route is the shortest one
    edges["rq_cost"] = edges["length"] * (1.0 + weight * edges["penalty"])
    return ox.graph_from_gdfs(nodes, edges)

G_weighted = add_costs()
print(f"{len(nodes):,} intersections · {len(edges):,} street segments · mean penalty {edges.penalty.mean():.2f}")

In [ ]:
SUMMARY_COLS = ['ndvi', 'stress_rate']

ORIG_XY = (433200, 5791200)    # south-west of the centre (EPSG:25832, metres)
DEST_XY = (436100, 5793000)    # north-east of the centre
ROUTES = {"Shortest": ("length", "blue"), "Greenest": ("rq_cost", "green")}

def edge_used(graph, u, v, weight):
    return min(graph[u][v].values(), key=lambda d: d[weight])        # cheapest of parallel edges

def compare_routes(graph, orig_xy=ORIG_XY, dest_xy=DEST_XY):
    o = ox.nearest_nodes(graph, X=orig_xy[0], Y=orig_xy[1])
    d = ox.nearest_nodes(graph, X=dest_xy[0], Y=dest_xy[1])
    rows, lines = [], []
    for name, (weight, colour) in ROUTES.items():
        path = nx.shortest_path(graph, o, d, weight=weight)           # Dijkstra
        segs = [edge_used(graph, u, v, weight) for u, v in zip(path[:-1], path[1:])]
        length = sum(s["length"] for s in segs)
        row = {"route": name, "length_km": length / 1000}
        for col in SUMMARY_COLS:                                       # length-weighted mean along the route
            row[col] = sum(s["length"] * s[col] for s in segs) / length
        rows.append(row)
        pts = [Point(graph.nodes[n]["x"], graph.nodes[n]["y"]) for n in path]
        lines.append({"Route": name, "Distance": f"{length/1000:.2f} km", "geometry": LineString(pts)})
    table = pd.DataFrame(rows).set_index("route")
    table["detour_%"] = (table.length_km / table.loc["Shortest", "length_km"] - 1) * 100
    m = gpd.GeoDataFrame(lines, crs=CRS).explore(column="Route", cmap=[c for _, c in ROUTES.values()],
                                                 categories=list(ROUTES), tiles=BASEMAP,
                                                 style_kwds=dict(weight=5), tooltip=["Route", "Distance"])
    folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m)
    folium.LayerControl().add_to(m)
    display(table.round(2))
    return m

compare_routes(G_weighted)

### ✏️ Your turn 6
With `weight = 1` the greenest route may be identical to (or only slightly different from) the shortest one.
**Increase the weight** (try `3`, then `10`) and compare again. How much detour (%) would *you* accept for a greener ride?

<details><summary>🆘 Hint</summary>

Write a number instead of `____`, e.g. `3`. Larger weight = grey streets look "longer" to the algorithm.
</details>

In [ ]:
G_weighted = add_costs(weight=____)
compare_routes(G_weighted)

> 💬 **Point for discussion:** Is the greenest route also calmer (compare `stress_rate`)? NDVI also counts private gardens behind walls — would street-view greenness (GVI) make a better routing cost?

### 📚 References
- Biljecki, F. & Ito, K. (2021). Street view imagery in urban analytics and GIS: A review. *Landscape and Urban Planning* 215, 104217.
- Cordts, M. et al. (2016). The Cityscapes Dataset for Semantic Urban Scene Understanding. *CVPR*.
- Getis, A. & Ord, J. K. (1992). The analysis of spatial association by use of distance statistics. *Geographical Analysis* 24(3), 189–206.
- Haklay, M. (2010). How good is volunteered geographical information? *Environment and Planning B* 37(4), 682–703.
- Jocher, G., Qiu, J. & Chaurasia, A. (2026). *Ultralytics YOLO26.* https://docs.ultralytics.com/models/yolo26
- Kyriakou, K., Resch, B. et al. (2019). Detecting Moments of Stress from Measurements of Wearable Physiological Sensors. *Sensors* 19(17), 3805.
- Li, X. et al. (2015). Assessing street-level urban greenery using Google Street View and a modified green view index. *Urban Forestry & Urban Greening* 14(3), 675–685.
- Lin, T.-Y. et al. (2014). Microsoft COCO: Common Objects in Context. *ECCV*.
- Openshaw, S. (1984). *The Modifiable Areal Unit Problem.* CATMOG 38.
- Redmon, J. et al. (2016). You Only Look Once: Unified, Real-Time Object Detection. *CVPR*.
- Tobler, W. (1970). A computer movie simulating urban growth in the Detroit region. *Economic Geography* 46, 234–240.
- Tucker, C. J. (1979). Red and photographic infrared linear combinations for monitoring vegetation. *Remote Sensing of Environment* 8, 127–150.
- Voogt, J. A. & Oke, T. R. (2003). Thermal remote sensing of urban climates. *Remote Sensing of Environment* 86, 370–384.
- Xie, E. et al. (2021). SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers. *NeurIPS*.
- Zha, Y., Gao, J. & Ni, S. (2003). Use of normalized difference built-up index in automatically mapping urban areas from TM imagery. *International Journal of Remote Sensing* 24(3), 583–594.

*Data: Sentinel-2 © Copernicus/ESA; Landsat courtesy of USGS; © OpenStreetMap contributors (ODbL); street view images © Mapillary contributors (CC-BY-SA).*